In [2]:
import numpy as np
import pandas as pd

In [3]:
ENTRY_AGE = 35
POLICY_TERM = 20
SUM_ASSURED = 1_000_000
INTEREST_RATE = 0.05
INFLATION_FACTOR = 0.05
v = 1 / (1 + INTEREST_RATE)

INITIAL_EXPENSE = 1000
RENEWAL_EXPENSE = 200
COMMISSION_RATE = 0.10
PREMIUM_COLLECTION_EXPENSE = 50
PREMIUM_PAYING_TERM = 20

ANNUAL_BONUS_RATE = 0.03

INITIAL_COMMISSION_RATE = COMMISSION_RATE
RENEWAL_COMMISSION_RATE = COMMISSION_RATE

MATURITY_BENEFIT = SUM_ASSURED

In [4]:
# 2. Load mortality dataset
mortality_data = pd.read_excel(
    "D:\Actuary\Projects Actuarial_Practice\Life_insurance_engine\data\mortality_tables\IIAM 12-15.xlsx",
)
mortality_data.head()

,Age,qx
0,20,0.000284
1,21,0.000305
2,22,0.000328
3,23,0.000353
4,24,0.000379


In [5]:
def mortality_table(ENTRY_AGE, POLICY_TERM, mortality_data):

    # Select mortality rates from dataset
    df = mortality_data[
        (mortality_data["Age"] >= ENTRY_AGE) &
        (mortality_data["Age"] < ENTRY_AGE + POLICY_TERM)
    ].copy()

    df = df.sort_values("Age").reset_index(drop=True)

    if len(df) != POLICY_TERM:
        raise ValueError("Mortality data is incomplete for the selected ages.")

    # Policy year
    df["Policy Year"] = range(1, POLICY_TERM + 1)

    # Individual probability of death
    df["Individual Probability of Death"] = df["qx"]

    # Initialize cohort
    lives_start = []
    deaths = []
    lives_end = []

    lives = 1.0

    for qx in df["qx"]:

        lives_start.append(lives)

        dx = lives * qx
        deaths.append(dx)

        lives = lives - dx
        lives_end.append(lives)

    # Add calculated columns
    df["Number of Lives at Start"] = lives_start
    df["Number of Deaths"] = deaths
    df["Number of Lives at End"] = lives_end

    return df[
        [
            "Policy Year",
            "Age",
            "Individual Probability of Death",
            "Number of Lives at Start",
            "Number of Deaths",
            "Number of Lives at End"
        ]
    ]

In [6]:
mortality = mortality_table(
    ENTRY_AGE=ENTRY_AGE,
    POLICY_TERM=POLICY_TERM,
    mortality_data=mortality_data
)

display(mortality.round(6))

,Policy Year,Age,Individual Probability of Death,Number of Lives at Start,Number of Deaths,Number of Lives at End
0,1,35,0.000847,1.000000,0.000847,0.999153
1,2,36,0.000913,0.999153,0.000912,0.998241
2,3,37,0.000984,0.998241,0.000982,0.997259
3,4,38,0.001061,0.997259,0.001058,0.996200
4,5,39,0.001144,0.996200,0.001140,0.995061
5,6,40,0.001234,0.995061,0.001228,0.993833
6,7,41,0.001332,0.993833,0.001324,0.992509
7,8,42,0.001438,0.992509,0.001427,0.991082
8,9,43,0.001553,0.991082,0.001539,0.989543
9,10,44,0.001679,0.989543,0.001661,0.987881


In [7]:
def calculate_benefit_epv(
    sum_assured,
    mortality,
    interest_rate
):
    v = 1 / (1 + interest_rate)
    benefit_epv = 0

    for k in range(len(mortality)):

        qx = mortality.loc[
            k, "Individual Probability of Death"
        ]

        kpx = mortality.loc[
            k, "Number of Lives at Start"
        ]

        benefit_epv += (
            sum_assured
            * (v ** (k + 1))
            * kpx
            * qx
        )

    return benefit_epv


benefit_epv = calculate_benefit_epv(
    SUM_ASSURED,
    mortality,
    INTEREST_RATE
)

print(f"EPV of Death Benefits: ₹{benefit_epv:,.2f}")

EPV of Death Benefits: ₹21,080.83


In [13]:
def calculate_premium_annuity_epv(
    mortality,
    interest_rate,
    premium_payment_term
):
    v = 1 / (1 + interest_rate)

    premium_annuity_epv = 0

    for k in range(premium_payment_term):

        kpx = mortality.loc[
            k, "Number of Lives at Start"
        ]

        premium_annuity_epv += (
            (v ** k) * kpx
        )

    return premium_annuity_epv


premium_annuity_epv = calculate_premium_annuity_epv(
    mortality,
    INTEREST_RATE,
    PREMIUM_PAYING_TERM
)

print(f"Premium Annuity EPV: {premium_annuity_epv:.6f}")

Premium Annuity EPV: 12.945298


In [14]:
def cashflow_projection(
    SUM_ASSURED,
    ANNUAL_PREMIUM,
    POLICY_TERM,
    PREMIUM_PAYING_TERM,
    ANNUAL_BONUS_RATE,
    MATURITY_BENEFIT= SUM_ASSURED
):

    CASHFLOWS = []
    ACCRUED_BONUS = 0

    for year in range(1, POLICY_TERM + 1):

        # 1. Premium
        premium = (
            ANNUAL_PREMIUM
            if year <= PREMIUM_PAYING_TERM
            else 0
        )

        # 2. Initial expenses
        initial_exp = (
            INITIAL_EXPENSE if year == 1 else 0
        )

        # 3. Inflation-adjusted renewal expenses
        renewal_exp = (
            RENEWAL_EXPENSE * (1 + INFLATION_FACTOR) ** (year - 1)
            if year > 1 else 0
        )

        # 4. Premium collection expenses
        collection_exp = (
            PREMIUM_COLLECTION_EXPENSE
            if premium > 0 else 0
        )

        # 5. Commission
        initial_comm = (
            premium * COMMISSION_RATE
            if year == 1 else 0
        )

        renewal_comm = (
            premium * COMMISSION_RATE
            if year > 1 else 0
        )

        # 6. Accrued bonus
        annual_bonus = SUM_ASSURED * ANNUAL_BONUS_RATE

        ACCRUED_BONUS += annual_bonus

        # 7. Death benefit
        base_death_benefit = SUM_ASSURED

        total_death_benefit = (
            base_death_benefit + ACCRUED_BONUS
        )

        # 8. Maturity benefit
        maturity = (
            MATURITY_BENEFIT + ACCRUED_BONUS
            if year == POLICY_TERM else 0
        )

        # 9. Discount factor
        discount_factor = v ** year

        # 10. Total expenses
        total_expenses = (
            initial_exp
            + renewal_exp
            + collection_exp
        )

        total_commission = (
            initial_comm + renewal_comm
        )

        # 11. Expected cash flow before claims
        net_cashflow_before_claims = (
            premium
            - total_expenses
            - total_commission
        )

        CASHFLOWS.append({

            "Policy Year": year,
            "Age": ENTRY_AGE + year - 1,

            "Premium": premium,

            "Initial Expenses": initial_exp,
            "Renewal Expenses": renewal_exp,
            "Premium Collection Expense": collection_exp,

            "Initial Commission": initial_comm,
            "Renewal Commission": renewal_comm,

            "Base Death Benefit": base_death_benefit,
            "Accrued Bonus": ACCRUED_BONUS,
            "Total Death Benefit": total_death_benefit,

            "Maturity Benefit": maturity,

            "Discount Factor": discount_factor,

            "Net Cash Flow Before Claims": net_cashflow_before_claims

        })

    return pd.DataFrame(CASHFLOWS)

In [15]:
def calculate_net_premium(
    benefit_epv,
    premium_annuity_epv
):
    return benefit_epv / premium_annuity_epv


annual_net_premium = calculate_net_premium(
    benefit_epv,
    premium_annuity_epv
)

print(f"Annual Net Premium: ₹{annual_net_premium:,.2f}")

Annual Net Premium: ₹1,628.45


In [18]:
cashflow = cashflow_projection(
    SUM_ASSURED=SUM_ASSURED,
    ANNUAL_PREMIUM=annual_net_premium,
    POLICY_TERM=POLICY_TERM,
    PREMIUM_PAYING_TERM=PREMIUM_PAYING_TERM,
    ANNUAL_BONUS_RATE=0.03,
    MATURITY_BENEFIT=0
)

display(cashflow.round(2))

,Policy Year,Age,Premium,Initial Expenses,Renewal Expenses,Premium Collection Expense,Initial Commission,Renewal Commission,Base Death Benefit,Accrued Bonus,Total Death Benefit,Maturity Benefit,Discount Factor,Net Cash Flow Before Claims
0,1,35,1628.45,1000,0.00,50,162.85,0.00,1000000,30000.0,1030000.0,0.0,0.95,415.61
1,2,36,1628.45,0,210.00,50,0.00,162.85,1000000,60000.0,1060000.0,0.0,0.91,1205.61
2,3,37,1628.45,0,220.50,50,0.00,162.85,1000000,90000.0,1090000.0,0.0,0.86,1195.11
3,4,38,1628.45,0,231.53,50,0.00,162.85,1000000,120000.0,1120000.0,0.0,0.82,1184.08
4,5,39,1628.45,0,243.10,50,0.00,162.85,1000000,150000.0,1150000.0,0.0,0.78,1172.51
5,6,40,1628.45,0,255.26,50,0.00,162.85,1000000,180000.0,1180000.0,0.0,0.75,1160.35
6,7,41,1628.45,0,268.02,50,0.00,162.85,1000000,210000.0,1210000.0,0.0,0.71,1147.59
7,8,42,1628.45,0,281.42,50,0.00,162.85,1000000,240000.0,1240000.0,0.0,0.68,1134.19
8,9,43,1628.45,0,295.49,50,0.00,162.85,1000000,270000.0,1270000.0,0.0,0.64,1120.12
9,10,44,1628.45,0,310.27,50,0.00,162.85,1000000,300000.0,1300000.0,0.0,0.61,1105.34
